# 低含水 log 局所混合版 cf×dry: ネストLOSO 検証と樹種別可視化

`make_submission.py` 現行(低log混合 ON)モデルの **honest なネストLOSO** 結果を可視化する。
ネスト本体は `cf4_dry_lowlog_nested_loso.py`(各外側fold=1樹種除外ごとに、残り樹種の内側LOSOで
全ハイパラ+低logゲート center_low/scale_low を Optuna 再選択 → 除外樹種を予測)。本ノートは保存済み
OOF(`cf4_dry_lowlog_nested_oof.npz`)と要約(`cf4_dry_lowlog_nested_result.json`)を読み込んで描画する。

> 楽観(全LOSO集計選択)では low-log macro=14.36(無し14.47)。ネストで +0.11 改善が残るかを検証。

In [ ]:
import json, numpy as np, pandas as pd
import matplotlib, matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = ['Hiragino Sans', 'AppleGothic', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False; matplotlib.rcParams['figure.dpi'] = 110

d = np.load('cf4_dry_lowlog_nested_oof.npz', allow_pickle=True)
cf, cfb, dry, ens, y, groups = d['cf'], d['cfb'], d['dry'], d['ens'], d['y'], d['groups']
species = np.unique(groups)
res = json.load(open('cf4_dry_lowlog_nested_result.json'))
macro = lambda p: np.mean([np.sqrt(np.mean((p[groups == s]-y[groups == s])**2)) for s in species])
overall = lambda p: np.sqrt(np.mean((p-y)**2))
print('=== 低log ネストLOSO(honest) ===')
print('macro=%.3f  overall=%.2f  ベイスギ=%.1f'
      % (res['macro'], res['overall'], res['bsg']))
print('\n参考(楽観・全LOSO集計): low-log macro=14.36 / 無し14.47 / base nested(log無し)=17.92')

In [ ]:
# 各fold で内側選択された 低logゲート center_low/scale_low
tab = pd.DataFrame(res['per_species']).sort_values('meanMC')
print(tab[['species','n','meanMC','outer_rmse','inner_best_macro','center_low','scale_low','center','scale','gate']]
      .round(1).to_string(index=False))

In [ ]:
# 図1: 樹種別 RMSE(ネスト) と 全体指標
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sp = tab['species'].values; xpos = np.arange(len(sp))
per = lambda p: [np.sqrt(np.mean((p[groups == s]-y[groups == s])**2)) for s in sp]
bw = 0.4
ax[0].bar(xpos-bw/2, per(cfb), bw, label='cf_blend', color='tab:blue')
ax[0].bar(xpos+bw/2, per(ens), bw, label='動的ens(低log)', color='crimson')
ax[0].set_xticks(xpos); ax[0].set_xticklabels(sp, rotation=45, ha='right', fontsize=8)
ax[0].set_ylabel('ネストLOSO RMSE'); ax[0].set_title('樹種別RMSE(平均MC昇順)'); ax[0].legend()
labels = ['cf_blend', 'dry', '動的ens']; vals = [macro(cfb), macro(dry), macro(ens)]
ax[1].bar(labels, vals, color=['tab:blue', 'tab:green', 'crimson'])
for i, v in enumerate(vals): ax[1].text(i, v, f'{v:.2f}', ha='center', va='bottom')
ax[1].set_ylabel('ネストLOSO macro-RMSE'); ax[1].set_title('macro(低いほど良)')
plt.tight_layout(); plt.show()

In [ ]:
# 図2: 樹種別 真値 vs ネストLOSO予測(平均MC昇順)
order_sp = tab['species'].values
preds = {'cf_blend': cfb, 'dry': dry, '動的ens': ens}
colors = {'cf_blend': 'tab:blue', 'dry': 'tab:green', '動的ens': 'crimson'}
ncol = 3; nrow = int(np.ceil(len(order_sp)/ncol))
fig, ax = plt.subplots(nrow, ncol, figsize=(15, 3.0*nrow)); ax = ax.ravel()
for i, s in enumerate(order_sp):
    idx = np.where(groups == s)[0]; o = idx[np.argsort(y[idx])]; x = np.arange(len(o)); a = ax[i]
    a.plot(x, y[o], '-o', color='k', ms=3, lw=1, label='真値', zorder=3)
    for mn_, p in preds.items(): a.plot(x, p[o], '.', color=colors[mn_], ms=5, alpha=.75, label=mn_)
    a.set_title('%s n=%d MC%.0f  ens-RMSE=%.0f' % (s, len(o), y[o].mean(),
                np.sqrt(np.mean((ens[o]-y[o])**2))), fontsize=9)
    a.set_xlabel('樹種内サンプル番号(MC昇順)'); a.set_ylabel('含水率 [%]')
for j in range(len(order_sp), len(ax)): ax[j].axis('off')
ax[0].legend(fontsize=8)
plt.suptitle('樹種別 真値 vs ネストLOSO予測(低log局所混合 cf×dry)', y=1.0)
plt.tight_layout(); plt.show()

## まとめ

- ネストLOSO は **ゲート/特徴しきい値/低logゲートを fold 内で再選択**するため、楽観バイアスを除いた honest 推定。
- 楽観で +0.11 だった低log の改善が、ネストでも残るか(ベイスギ以外の中低MC樹種で過大評価が緩和されているか)を
  樹種別図で確認する。ベイスギは学習唯一の超高MC樹種ゆえネストでは構造的に大崩れする(base同様)点に注意。